# LC 57 — Insert Interval
**Day 51 | Theme: Intervals | Difficulty: Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Three-phase linear scan — no sort
needed because the list is already sorted. Copy intervals that
end before the new one starts. Merge all that overlap with the
new interval. Copy the rest unchanged.
</div>

## Official Problem Statement

You are given an array of non-overlapping intervals `intervals`
sorted in ascending order by `starti`, and an interval
`newInterval = [start, end]`.

Insert `newInterval` into `intervals` such that `intervals` is
still sorted in ascending order and still has no overlapping
intervals (merge if necessary).

Return `intervals` after the insertion.

**Constraints:**
- `0 <= intervals.length <= 10^4`
- `intervals[i].length == 2`
- `0 <= starti <= endi <= 10^5`
- `intervals` is sorted by `starti` in ascending order
- `0 <= start <= end <= 10^5`

## What This Is Actually Asking

The existing interval list is already clean (sorted, no overlaps).
You need to insert one new interval and restore that clean state.
The new interval may overlap zero, one, or many existing ones.
Because the list is pre-sorted you do not need to sort again —
a single left-to-right scan handles the three possible regions.

## Walk Through an Example by Hand

Input: `intervals=[[1,2],[3,5],[6,7],[8,10],[12,16]]`,
`newInterval=[4,8]`

**Phase 1 — Copy intervals ending before new starts (end < 4):**
- [1,2]: 2 < 4 → copy. result = [[1,2]]
- [3,5]: 5 < 4? No → stop Phase 1. i=1

**Phase 2 — Merge overlapping (start <= 8):**
- [3,5]: 3 <= 8 → overlap! new=[min(4,3),max(8,5)]=[3,8]
- [6,7]: 6 <= 8 → overlap! new=[min(3,6),max(8,7)]=[3,8]
- [8,10]: 8 <= 8 → overlap! new=[min(3,8),max(8,10)]=[3,10]
- [12,16]: 12 <= 8? No → stop Phase 2. i=4
- Append merged new=[3,10]. result=[[1,2],[3,10]]

**Phase 3 — Copy remaining:**
- [12,16] → copy. result=[[1,2],[3,10],[12,16]]

**Output:** `[[1,2],[3,10],[12,16]]`

## The Picture

```
Timeline axis 0..16

Existing intervals:
  [1,2]   ||
  [3,5]      |==|
  [6,7]           ||
  [8,10]             |==|
  [12,16]                    |====|

New interval to insert:
  [4,8]        |======|

  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16

Three phases:
  Phase 1: [1,2] ends at 2 < 4 (new start) → COPY as-is
  Phase 2: [3,5],[6,7],[8,10] all start <= 8 (new end) → MERGE
            merged new = [3, 10]
  Phase 3: [12,16] starts > 10 (merged end) → COPY as-is

Result:
  [1,2]   ||
  [3,10]     |===========|
  [12,16]                    |====|

  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16
```

## When To Use This Pattern

- When inserting into a pre-sorted, non-overlapping list, think
  **three-phase scan, no re-sort needed**.
- When a problem says "already sorted" intervals, think **exploit
  the sort, walk linearly**.
- When merging a new item with an existing ordered set, think
  **phase 1: before, phase 2: overlap, phase 3: after**.
- When you see "insert and maintain clean state", think
  **expand bounds during the overlap phase**.
- When the input is guaranteed clean, think **no need for
  O(n log n) sort — O(n) is achievable**.

## The Approach

Walk the sorted list with index `i` and an output list `result`.
Phase 1: while the current interval ends strictly before
`newInterval` starts, copy it to result and advance `i`.
Phase 2: while the current interval starts at or before
`newInterval`'s end, they overlap — expand `newInterval` to
cover both, then advance `i`. Append the expanded interval.
Phase 3: copy all remaining intervals unchanged.

In [ ]:
from typing import List

In [ ]:
def test_harness(func):
    """Run test cases for insert_interval."""
    def norm(intervals):
        """Sort for stable comparison."""
        return sorted([sorted(iv) for iv in intervals])

    cases = [
        # (intervals, newInterval, expected)
        ([[1,3],[6,9]], [2,5], [[1,5],[6,9]]),
        (
            [[1,2],[3,5],[6,7],[8,10],[12,16]],
            [4,8],
            [[1,2],[3,10],[12,16]]
        ),
        ([], [5,7], [[5,7]]),         # empty list
        ([[1,5]], [2,3], [[1,5]]),    # new contained inside
        ([[1,5]], [0,0], [[0,0],[1,5]]),  # insert before all
        ([[1,5]], [6,8], [[1,5],[6,8]]),  # insert after all
        ([[1,5]], [5,7], [[1,7]]),    # touching edge merges
        ([[1,5]], [0,10], [[0,10]]),  # new covers everything
    ]

    passed = 0
    for i, (intervals, new_iv, expected) in enumerate(cases):
        ivs_copy = [iv[:] for iv in intervals]
        result = func(ivs_copy, new_iv[:])
        if norm(result) == norm(expected):
            print(f"  Case {i+1}: PASSED")
            passed += 1
        else:
            print(f"  Case {i+1}: FAILED")
            print(f"    intervals: {intervals}")
            print(f"    newInterval: {new_iv}")
            print(f"    Expected: {expected}")
            print(f"    Got:      {result}")

    total = len(cases)
    print(f"\nResult: {passed}/{total} passed")
    if passed == total:
        print("All tests PASSED!")
    else:
        print(f"{total - passed} test(s) FAILED.")

In [ ]:
def insert(
    intervals: List[List[int]],
    newInterval: List[int]
) -> List[List[int]]:
    """
    Insert newInterval into sorted non-overlapping intervals.

    Strategy (three phases, single O(n) pass):
        Phase 1: Copy all intervals ending before new starts.
                 Condition: intervals[i][1] < newInterval[0]
        Phase 2: Merge all overlapping with new.
                 Condition: intervals[i][0] <= newInterval[1]
                 Expand: new = [min(new[0], iv[0]),
                                max(new[1], iv[1])]
        Phase 3: Copy all remaining intervals.

    Args:
        intervals:   Sorted, non-overlapping list of [s,e] pairs.
        newInterval: The interval to insert and merge.

    Returns:
        New sorted, non-overlapping list after insertion.

    Time:  O(n)
    Space: O(n) for output
    """
    result = []
    i = 0
    n = len(intervals)

    print(f"[DEBUG] intervals={intervals}")
    print(f"[DEBUG] newInterval={newInterval}")

    # TODO: Phase 1 — copy non-overlapping before new
    print(f"[DEBUG] after phase 1: result={result}, i={i}")

    # TODO: Phase 2 — merge overlapping into new
    print(f"[DEBUG] merged newInterval={newInterval}")

    # TODO: Phase 3 — copy remaining
    pass


# Quick smoke test
sample_ivs = [[1,3],[6,9]]
sample_new = [2,5]
print(f"[DEBUG] insert({sample_ivs},{sample_new}) = "
      f"{insert(sample_ivs, sample_new)}")

In [ ]:
# Uncomment and run when solution is ready
# test_harness(insert)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Naive: insert + re-sort + merge | O(n log n) | O(n) | Wastes the pre-sort |
| Three-phase linear scan (optimal) | O(n) | O(n) | Exploits sorted order |

Because the input is guaranteed sorted, we avoid any sort step.
Every interval is visited exactly once, giving true O(n) time.

## Real World Connection

At **Citi**, inserting a new trading halt window into an existing
compliance schedule (which is already sorted) must not disrupt
surrounding windows — this three-phase insert maintains the
invariant without reprocessing the entire calendar.
On **AWS**, appending a new maintenance window to a pre-ordered
EC2 downtime schedule uses exactly this logic to merge with any
adjacent windows efficiently.
In **data engineering**, streaming pipelines that receive out-of-
order micro-batch time ranges insert each new range into an
ordered active-window list — this pattern is the core of that
reconciliation step.
Any system maintaining a sorted event timeline benefits from this
O(n) insert-and-merge over a naive re-sort approach.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra